In [1]:
from functools import partial
from notebooks._utils import report_series_ensemble_accuracy_by_nparas
from notebooks._utils import calculate_series_ensemble_accuracy
from notebooks._utils import calculate_parallel_ensemble_accuracy


ds_name = "myriadlama"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

In [2]:
def get_layers(model: str):
    if model.startswith("llama3.2_1b"):
        layers = 12
    elif model.startswith("llama3.2_3b"):
        layers = 21
    elif model.startswith("llama3.1_8b"):
        layers = 24
    elif model.startswith("qwen2.5_3b"):
        layers = 27
    elif model.startswith("qwen2.5_7b"):
        layers = 21
    elif model.startswith("qwen2.5_14b"):
        layers = 36
    else:
        raise NotImplementedError(f"Layers not defined for model {model}")
    return layers

In [4]:
# for model_name in ["llama3.2_1b", "llama3.2_1b_it", "llama3.2_3b", "llama3.2_3b_it", "llama3.1_8b", "llama3.1_8b_it", "qwen2.5_3b", "qwen2.5_3b_it", "qwen2.5_7b", "qwen2.5_7b_it", "qwen2.5_14b", "qwen2.5_14b_it"]:
for model_name in ["llama3.2_1b", "llama3.2_1b_it", "llama3.2_3b", "llama3.2_3b_it", "qwen2.5_3b", "qwen2.5_3b_it"]:
# for model_name in ["llama3.2_1b"]:
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    
    report_accuracy = partial(
        report_series_ensemble_accuracy_by_nparas, 
        dump_file_prefix=dump_file_prefix,
        single_para_qapair=True,
        explicit_prompts=False,
        repeat_paras=False)
    
    for num_fewshots in range(10):
        print(f"\n---- {num_fewshots}-shots: Calculating baseline ----")
        df = calculate_series_ensemble_accuracy(
            dump_file_prefix=dump_file_prefix, 
            single_para_qapair=True, explicit_prompts=False, repeat_paras=False, 
            modifyattn=False, modifyrope=False, scale_score=0, 
            num_paraphrases=1, num_fewshots=num_fewshots)

        # print(f"---- {num_fewshots}-shots: ❌ attention mask / ❌ rope modifications / ❌ scale score ----")
        # report_accuracy(modifyattn=False, modifyrope=False, scale_score=False, num_fewshots=num_fewshots)
        # print("\n---- ❌ Attention mask / ✅ rope modifications / ❌ scale score ----")
        # report_accuracy(modifyattn=False, modifyrope=True, scale_score=False)
        # print("\n---- ✅ Attention mask / ❌ rope modifications / ❌ scale score ----")
        # report_accuracy(modifyattn=True, modifyrope=False, scale_score=False)
        # print("\n---- ✅ Attention mask / ✅ Rope modifications / ❌ scale score ----")
        # report_accuracy(modifyattn=True, modifyrope=True, scale_score=False, num_fewshots=num_fewshots)
        print(f"---- {num_fewshots}-shots: ✅ Attention mask / ✅ Rope modifications / ✅ Scale score ----")
        report_accuracy(modifyattn=True, modifyrope=True, scale_score=True, num_fewshots=num_fewshots)
        
        # print(f"---- {num_fewshots}-shots: Parallel Average ----")
        # calculate_parallel_ensemble_accuracy(
        #     dump_file_prefix=dump_file_prefix, 
        #     repeat_paras=False,
        #     logits_ensemble_method="avg",
        #     num_paraphrases=5, num_fewshots=num_fewshots, use_generation=True)

        print(f"---- {num_fewshots}-shots: Parallel Average (+Layer avg) ----")
        calculate_parallel_ensemble_accuracy(
            dump_file_prefix=dump_file_prefix, 
            repeat_paras=False,
            logits_ensemble_method="avg",
            num_paraphrases=5, num_fewshots=num_fewshots, use_generation=True, 
            ensemble_method="layer_output_avg", multilayer=True, 
            ensemble_layer=get_layers(model_name), token_mode="last", ensemble_alpha=1)

        # print(f"---- {num_fewshots}-shots: Parallel Maximum ----")
        # calculate_parallel_ensemble_accuracy(
        #     dump_file_prefix=dump_file_prefix, 
        #     repeat_paras=False,
        #     logits_ensemble_method="max",
        #     num_paraphrases=5, num_fewshots=num_fewshots, use_generation=True)

        # print(f"---- {num_fewshots}-shots: Parallel Maximum (+Layer avg) ----")
        # calculate_parallel_ensemble_accuracy(
        #     dump_file_prefix=dump_file_prefix, 
        #     repeat_paras=False,
        #     logits_ensemble_method="max",
        #     num_paraphrases=5, num_fewshots=num_fewshots, use_generation=True, 
        #     ensemble_method="layer_output_avg", multilayer=True,
        #     ensemble_layer=get_layers(model_name), token_mode="last", ensemble_alpha=1)

        


=================== Model: llama3.2_1b ===================

---- 0-shots: Calculating baseline ----
Acc: 0.1203 ==> 🏷️ 1paras 0shots 1QA      (Baseline)
---- 0-shots: ✅ Attention mask / ✅ Rope modifications / ✅ Scale score ----
Acc: 0.1263 ==> 🏷️ 5paras 0shots 1QA   +Attn +Rope +ScaleScore
---- 0-shots: Parallel Average (+Layer avg) ----
Acc: 0.1611 ==> 🏷️ 5paras 0shots  layer_output_avg layer12 Multilayer alpha1 token-last

---- 1-shots: Calculating baseline ----
Acc: 0.2381 ==> 🏷️ 1paras 1shots 1QA      (Baseline)
---- 1-shots: ✅ Attention mask / ✅ Rope modifications / ✅ Scale score ----
Acc: 0.3258 ==> 🏷️ 5paras 1shots 1QA   +Attn +Rope +ScaleScore
---- 1-shots: Parallel Average (+Layer avg) ----
Acc: 0.3247 ==> 🏷️ 5paras 1shots  layer_output_avg layer12 Multilayer alpha1 token-last

---- 2-shots: Calculating baseline ----
Acc: 0.2887 ==> 🏷️ 1paras 2shots 1QA      (Baseline)
---- 2-shots: ✅ Attention mask / ✅ Rope modifications / ✅ Scale score ----
Acc: 0.3800 ==> 🏷️ 5paras 2shots 